# 01 — Corpus Collection
Walks the repository and collects every source of ARO knowledge into a single
manifest. Calls the `aro` CLI to capture live grammar, action, and example data.

**Output:** `../data/01_corpus/manifest.json`

In [ ]:
import sys, importlib
from pathlib import Path
_cfg_dir = Path('.').resolve()
if _cfg_dir not in [Path(p) for p in sys.path]:
    sys.path.insert(0, str(_cfg_dir))
import config; importlib.reload(config); from config import *
print(f'Config loaded | model={MODEL_ID}')
print(f'ARO root:  {ARO_ROOT}')
print(f'Pairs:     {PAIRS_FILE}')
print(f'Adapters:  {ADAPTER_DIR}')

import os
import json
import subprocess
from pathlib import Path

ARO_APP_DIR = (SCRIPT_DIR / '../../../ARO-Application/').resolve()
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f'ARO root:        {ARO_ROOT}')
print(f'ARO-Application: {ARO_APP_DIR}  (exists: {ARO_APP_DIR.exists()})')
print(f'Output:          {DATA_DIR}')

## Check aro CLI

In [ ]:
def run_aro(args, timeout=30):
    try:
        r = subprocess.run(['aro'] + args, capture_output=True, text=True, timeout=timeout)
        return r.stdout.strip(), r.stderr.strip(), r.returncode
    except FileNotFoundError:
        return '', 'aro binary not found in PATH', -1
    except subprocess.TimeoutExpired:
        return '', 'timeout', -1

out, err, code = run_aro(['--version'])
print(f'aro: {out or err}')

## Collect .aro files

In [ ]:
aro_files = []
search_roots = [ARO_ROOT / 'Examples']
if ARO_APP_DIR.exists():
    search_roots.append(ARO_APP_DIR)

for root in search_roots:
    found = list(root.rglob('*.aro'))
    print(f'  {root.name}: {len(found)} .aro files')
    aro_files.extend(found)

print(f'Total: {len(aro_files)} .aro files')

## Collect Proposals, Book, Swift action files

In [ ]:
proposals     = sorted((ARO_ROOT / 'Proposals').glob('*.md'))
book_chapters = sorted((ARO_ROOT / 'Book').rglob('*.md')) if (ARO_ROOT / 'Book').exists() else []
swift_actions = sorted((ARO_ROOT / 'Sources' / 'ARORuntime' / 'Actions').rglob('*.swift'))

print(f'Proposals:     {len(proposals)}')
print(f'Book chapters: {len(book_chapters)}')
print(f'Swift actions: {len(swift_actions)}')

## Fetch live data from aro CLI

In [ ]:
print('Calling aro CLI...')
syntax_out,   _, _ = run_aro(['syntax'])
actions_out,  _, _ = run_aro(['actions'])
examples_out, _, _ = run_aro(['examples'])

print(f'aro syntax:   {len(syntax_out):,} chars')
print(f'aro actions:  {len(actions_out):,} chars')
print(f'aro examples: {len(examples_out):,} chars')

## Read static reference docs

In [ ]:
extra_docs = {}
for doc_path in [ARO_ROOT / 'README.md', ARO_ROOT / 'CLAUDE.md', ARO_ROOT / 'OVERVIEW.md']:
    if doc_path.exists():
        extra_docs[doc_path.name] = doc_path.read_text()
        print(f'  {doc_path.name}: {len(extra_docs[doc_path.name]):,} chars')

## Save manifest

In [ ]:
manifest = {
    'aro_files':     [str(f) for f in aro_files],
    'proposals':     [str(f) for f in proposals],
    'book_chapters': [str(f) for f in book_chapters],
    'swift_actions': [str(f) for f in swift_actions],
    'extra_docs':    {k: v[:60_000] for k, v in extra_docs.items()},
    'aro_syntax':    syntax_out,
    'aro_actions':   actions_out,
    'aro_examples':  examples_out,
}

manifest_path = DATA_DIR / 'manifest.json'
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2)

total = len(aro_files) + len(proposals) + len(book_chapters) + len(swift_actions)
print(f'Manifest saved: {manifest_path}')
print(f'Total files indexed: {total}')